# ЦФТ, отчёт коллег за август и Альфа: сверка и что такое ID в выгрузке ЦФТ

Четыре источника:
- **ЦФТ на 01.09** - выгрузка коллег `CFT_clients_010926.xlsx` (лежит рядом с тетрадкой);
- **отчёт за август** - `Data/08_Август_2026.xlsx`;
- **final_df за август** - сборка из Альфы за месяц;
- **моя выгрузка на 01.09** - `trade_acquiring_clients_2026-09-01.xlsx`, срез на дату.

Отчёт и final_df - за месяц: в них есть договоры, закончившиеся в течение августа. Моя выгрузка и ЦФТ - на 01.09. Поэтому в отчёте договоров больше, и секция 3 показывает, за счёт чего.

Секция 4 определяет, что такое колонка `id` в выгрузке ЦФТ. Мой «ЦФТ ID» - это ID **клиента** в ЦФТ (`c_cl_org`), поэтому с `id` он не совпал. Проверяются: «ID договора» в отчёте, `agr_id` в final_df (= `ods.scd1_z_r2_ip_merchants.id`), `n_agr` (ID договора в Альфе) и ID клиента. Плюс один запрос к озеру по выборке ID.


In [ ]:
import getpass
import re
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 240)
pd.set_option('display.max_colwidth', 80)

AS_OF_DATE = '2026-09-01'
REPORT_MONTH = '2026-08'
BASE_DIR = Path('/home/jovyan/documents/Equaring')
DATA_DIR = BASE_DIR / 'Data'
CFT_FILE = Path('CFT_clients_010926.xlsx').resolve()  # файл лежит рядом с тетрадкой
CFT_SHEET = 0
EX_FILE = DATA_DIR / '08_Август_2026.xlsx'
EX_SHEET = 0
FINAL_DF_CSV = DATA_DIR / 'final_df_period_2026_01_2026_08_final_df_script_3_1.csv'
MINE_FILE = BASE_DIR / ('trade_acquiring_clients_%s.xlsx' % AS_OF_DATE)
OUT_FILE = BASE_DIR / ('te_cft_report_alpha_%s.xlsx' % AS_OF_DATE)

# Колонки отчёта. None - найти автоматически, иначе точное имя колонки
EX_ID_COL = None
EX_NUM_COL = None
EX_INN_COL = None
EX_NAME_COL = None

LOOKUP_LAKE = True  # проверить ID ЦФТ по ods.scd1_z_r2_ip_merchants
LAKE_SAMPLE = 500   # сколько ID ЦФТ взять в выборку

as_of = pd.Timestamp(AS_OF_DATE)
as_of_ru = as_of.strftime('%d.%m.%Y')
MINE_SHEET = 'С ТЭ на %s' % as_of_ru
for p in (CFT_FILE, EX_FILE, FINAL_DF_CSV, MINE_FILE):
    print('есть ' if p.exists() else 'НЕТ  ', p)


## 1. Загрузка

In [ ]:
LEGAL_FORMS = [
    'ОБЩЕСТВО С ОГРАНИЧЕННОЙ ОТВЕТСТВЕННОСТЬЮ', 'НЕПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО', 'ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО',
    'АКЦИОНЕРНОЕ ОБЩЕСТВО', 'ИНДИВИДУАЛЬНЫЙ ПРЕДПРИНИМАТЕЛЬ', 'ГЛАВА КФХ',
    'ООО', 'ОАО', 'ЗАО', 'ПАО', 'НАО', 'АО', 'ИП', 'МБУ', 'МАУ', 'МБОУ', 'МАОУ', 'МБДОУ', 'МАДОУ', 'МКУ', 'ГБУ',
    'ГАУ', 'ГКУ', 'ГБУЗ', 'ФГБУ', 'ФГУП', 'МУП', 'ГУП', 'АНО', 'ТСЖ', 'СНТ', 'КФХ',
]
LEGAL_RE = re.compile(r'\b(%s)\b' % '|'.join(LEGAL_FORMS))
EMPTY = ('', 'NONE', 'NAN', '<NA>', 'NAT', 'NULL')

def _low(v):
    return re.sub(r'\s+', ' ', str(v)).strip().lower()

def norm_agr(v):
    if v is None or pd.isna(v):
        return None
    s = str(v).strip()
    if re.fullmatch(r'\d+\.0+', s):
        s = s.split('.')[0]
    s = re.sub(r'\s+', '', s).upper()
    return None if s in EMPTY else s

def norm_id(v):
    k = norm_agr(v)
    if not k:
        return None
    return (k.lstrip('0') or '0') if k.isdigit() else k

def norm_inn(v):
    if v is None or pd.isna(v):
        return None
    s = re.sub(r'\D', '', str(v).split('.')[0])
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    return s if len(s) in (10, 12) else None

def norm_name(v):
    if v is None or pd.isna(v):
        return None
    s = str(v).upper().replace('Ё', 'Е')
    s = re.sub(r'[^0-9A-ZА-Я]+', ' ', s)
    s = LEGAL_RE.sub(' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return None if s in EMPTY else s

def to_date(s):
    s = pd.Series(s, dtype=object)
    iso = s.map(lambda v: isinstance(v, str) and bool(re.match(r'\s*\d{4}-\d{2}-\d{2}', v)))
    out = pd.to_datetime(s.where(~iso), errors='coerce', dayfirst=True)
    out[iso] = pd.to_datetime(s[iso].str.strip().str[:10], errors='coerce', format='%Y-%m-%d')
    return out.dt.normalize()

def read_with_header(path, sheet, need=('договор',), max_rows=20):
    raw = pd.read_excel(path, sheet_name=sheet, header=None, dtype=object)
    for i in range(min(max_rows, len(raw))):
        cells = ' | '.join(_low(v) for v in raw.iloc[i] if pd.notna(v))
        if all(k in cells for k in need):
            names, seen = [], {}
            for j, v in enumerate(raw.iloc[i]):
                n = re.sub(r'\s+', ' ', str(v)).strip() if pd.notna(v) else 'col_%s' % j
                seen[n] = seen.get(n, 0) + 1
                names.append(n if seen[n] == 1 else '%s_%s' % (n, seen[n]))
            df = raw.iloc[i + 1:].copy()
            df.columns = names
            return df.dropna(how='all').reset_index(drop=True), i
    raise ValueError('Не нашёл строку заголовков с %s в первых %s строках %s' % (need, max_rows, path))

def pick(cols, exact, contains=None, exclude=()):
    low = {c: _low(c) for c in cols}
    for e in exact:
        for c in cols:
            if low[c] == e:
                return c
    if contains:
        for c in cols:
            if all(k in low[c] for k in contains) and not any(x in low[c] for x in exclude):
                return c
    return None

def digits_share(s):
    v = s.dropna().map(lambda x: re.sub(r'\.0+$', '', str(x).strip()))
    v = v[v != '']
    return float(v.str.fullmatch(r'\d{6,}').mean()) if len(v) else 0.0

def col_or_none(df, col, fn=None):
    if col is None:
        return pd.Series(None, index=df.index, dtype=object)
    s = df[col]
    return s.map(fn) if fn else s.astype('string').str.strip()


In [ ]:
cft_raw, _ = read_with_header(CFT_FILE, CFT_SHEET)
ccols = list(cft_raw.columns)
C = {
    'name': pick(ccols, ['наименование клиента'], ('наимен',), exclude=('тариф',)),
    'agr': pick(ccols, ['номер договора'], ('номер', 'договор'), exclude=('тариф', 'индивид')),
    'status': pick(ccols, ['статус'], ('статус',)),
    'branch': pick(ccols, ['код филиала'], ('филиал',)),
    'end': pick(ccols, ['дата окончания действия договора'], ('окончан',), exclude=('тариф',)),
    'id': pick(ccols, ['id']),
}
print('Колонки ЦФТ:', C)
if C['agr'] is None or C['id'] is None:
    raise KeyError('В выгрузке ЦФТ не нашлась колонка номера договора или id. Колонки: %s' % ccols)
cft = pd.DataFrame({
    'num_key': cft_raw[C['agr']].map(norm_agr),
    'Номер договора (ЦФТ)': col_or_none(cft_raw, C['agr']),
    'Клиент (ЦФТ)': col_or_none(cft_raw, C['name']),
    'Статус (ЦФТ)': col_or_none(cft_raw, C['status']),
    'Код филиала (ЦФТ)': col_or_none(cft_raw, C['branch']),
    'Дата окончания (ЦФТ)': to_date(cft_raw[C['end']]) if C['end'] else pd.NaT,
    'id': col_or_none(cft_raw, C['id'], norm_id),
    'ID (ЦФТ)': col_or_none(cft_raw, C['id']),
})
cft['name_n'] = cft['Клиент (ЦФТ)'].map(norm_name)
cft['inn_n'] = None
cft = cft[cft['num_key'].notna()].drop_duplicates('num_key').reset_index(drop=True)
print('ЦФТ: договоров', len(cft), '| уникальных id', cft['id'].nunique(),
      '| id из одних цифр: %.3f' % digits_share(cft['ID (ЦФТ)']))


In [ ]:
rep_raw, rep_hdr = read_with_header(EX_FILE, EX_SHEET, need=('инн', 'договор'))
cols = list(rep_raw.columns)
print('Отчёт: строка заголовка', rep_hdr, '| строк', len(rep_raw))
print('Колонки отчёта:', cols)

ex_id = EX_ID_COL or pick(cols, ['id договора', 'agr_id', 'abs_agr_id'], ('id', 'договор'))
ex_num = EX_NUM_COL or pick(cols, ['номер договора', '№ договора'], ('номер', 'договор'), exclude=('id',))
ex_inn = EX_INN_COL or pick(cols, ['инн', 'инн клиента'], ('инн',), exclude=('банк',))
ex_name = EX_NAME_COL or pick(
    cols,
    ['наименование клиента', 'наименование организации', 'наименование', 'название организации',
     'клиент', 'организация', 'наименование тсп'],
    ('наимен',), exclude=('тариф', 'филиал', 'всп', 'отдел', 'рф', 'mcc', 'мсс', 'банк'))
if ex_num and not EX_NUM_COL and digits_share(rep_raw[ex_num]) > 0.95 and ex_id is None:
    print('«%s» содержит только цифры - это ID договора, а не номер' % ex_num)
    ex_id, ex_num = ex_num, None
print('Колонки отчёта: ID = %r | номер = %r | ИНН = %r | наименование = %r' % (ex_id, ex_num, ex_inn, ex_name))
if ex_num is None:
    raise KeyError('В отчёте не нашлась колонка номера договора. Задайте EX_NUM_COL')

rep = pd.DataFrame({
    'id': col_or_none(rep_raw, ex_id, norm_id),
    'ID договора (отчёт)': col_or_none(rep_raw, ex_id),
    'num_key': col_or_none(rep_raw, ex_num, norm_agr),
    'Номер договора (отчёт)': col_or_none(rep_raw, ex_num),
    'inn_n': col_or_none(rep_raw, ex_inn, norm_inn),
    'ИНН (отчёт)': col_or_none(rep_raw, ex_inn),
    'Наименование (отчёт)': col_or_none(rep_raw, ex_name),
})
rep['name_n'] = rep['Наименование (отчёт)'].map(norm_name)
rep_rows = len(rep)
rep = rep[rep['num_key'].notna()].drop_duplicates('num_key').reset_index(drop=True)
print('Отчёт: строк %s, уникальных номеров %s' % (rep_rows, len(rep)))

fd_need = ['report_month', 'agr_id', 'n_agr', 'contract_number', 'inn', 'company_name', 'd_valid_from', 'd_valid_to']
fd_all = pd.read_csv(FINAL_DF_CSV, dtype=str, low_memory=False, usecols=lambda c: c in fd_need)
fd_m = fd_all[fd_all['report_month'].astype('string').str.strip().str[:7] == REPORT_MONTH].copy()
if fd_m.empty:
    raise ValueError('В final_df нет месяца %s' % REPORT_MONTH)
for c in fd_need:
    if c not in fd_m.columns:
        fd_m[c] = None
fd = pd.DataFrame({
    'id': fd_m['agr_id'].map(norm_id),
    'n_agr_id': fd_m['n_agr'].map(norm_id),
    'agr_id (final_df)': fd_m['agr_id'],
    'n_agr (final_df)': fd_m['n_agr'],
    'num_key': fd_m['contract_number'].map(norm_agr),
    'Номер договора (final_df)': fd_m['contract_number'],
    'inn_n': fd_m['inn'].map(norm_inn),
    'Наименование (final_df)': fd_m['company_name'],
    'Начало (final_df)': to_date(fd_m['d_valid_from']),
    'Окончание (final_df)': to_date(fd_m['d_valid_to']),
})
fd['name_n'] = fd['Наименование (final_df)'].map(norm_name)
fd = fd.dropna(subset=['id']).drop_duplicates('id').reset_index(drop=True)
fd_by_num = fd.dropna(subset=['num_key']).drop_duplicates('num_key').set_index('num_key')
print('final_df %s: договоров (agr_id) %s' % (REPORT_MONTH, len(fd)))

mine_raw = pd.read_excel(MINE_FILE, sheet_name=MINE_SHEET, dtype=object)
mine = pd.DataFrame({
    'num_key': mine_raw['Номер договора'].map(norm_agr),
    'Номер договора (мой)': mine_raw['Номер договора'].astype('string'),
    'inn_n': mine_raw['ИНН'].map(norm_inn),
    'ЦФТ ID': mine_raw['ЦФТ ID'].astype('string'),
    'client_id': mine_raw['ЦФТ ID'].map(norm_id),
    'Дата открытия (мой)': to_date(mine_raw['Дата открытия договора']),
})
mine = mine[mine['num_key'].notna()].drop_duplicates('num_key').reset_index(drop=True)
mine['id'] = mine['num_key'].map(fd_by_num['id'])
mine['name_n'] = mine['num_key'].map(fd_by_num['name_n'])
print('Моя выгрузка: договоров %s, нашлись в final_df по номеру %s' % (len(mine), int(mine['id'].notna().sum())))


## 2. Сколько и что с чем совпадает

«% A найдено в B» - доля ключей первого источника, которые есть во втором. У ЦФТ нет ИНН. ID договора у моей выгрузки взят из final_df по номеру.


In [ ]:
SRC = {
    'ЦФТ на 01.09': cft,
    'Отчёт за август': rep,
    'final_df за август': fd,
    'Моя выгрузка на 01.09': mine,
}
KEYS = [('номер', 'num_key'), ('ИНН', 'inn_n'), ('наименование', 'name_n'), ('ID договора', 'id')]

def keyset(name, col):
    if name == 'ЦФТ на 01.09' and col == 'id':
        return set()
    return set(SRC[name][col].dropna())

counts = pd.DataFrame([
    (name, len(df), df['num_key'].nunique(), df['inn_n'].nunique() or None, df['name_n'].nunique())
    for name, df in SRC.items()
], columns=['Источник', 'Договоров', 'Уникальных номеров', 'Клиентов по ИНН', 'Клиентов по наименованию'])

PAIRS = [
    ('Моя выгрузка на 01.09', 'Отчёт за август'),
    ('final_df за август', 'Отчёт за август'),
    ('ЦФТ на 01.09', 'Отчёт за август'),
    ('ЦФТ на 01.09', 'Моя выгрузка на 01.09'),
    ('ЦФТ на 01.09', 'final_df за август'),
]
rows = []
for a, b in PAIRS:
    for key, col in KEYS:
        sa, sb = keyset(a, col), keyset(b, col)
        if not sa or not sb:
            continue
        both = len(sa & sb)
        rows.append({'A': a, 'B': b, 'Ключ': key, 'У A': len(sa), 'У B': len(sb), 'Общих': both,
                     'Только A': len(sa - sb), 'Только B': len(sb - sa),
                     '% A найдено в B': round(both / len(sa) * 100, 1),
                     '% B найдено в A': round(both / len(sb) * 100, 1)})
overlap = pd.DataFrame(rows)
display(counts)
display(overlap)


## 3. Моя выгрузка на 01.09 и отчёт за август: откуда разница

- **Есть в отчёте, нет у меня.** Если по final_df договор закончился в августе - он был клиентом в августе, но на 01.09 уже нет. Это ожидаемая разница месячного отчёта и среза на дату.
- **Есть у меня, нет в отчёте.** Договоры, открытые 01.09, в августовский отчёт попасть не могут.


In [ ]:
rep_nums, my_nums = set(rep['num_key']), set(mine['num_key'])
rep_inns, my_inns = set(rep['inn_n'].dropna()), set(mine['inn_n'].dropna())

only_rep = rep[~rep['num_key'].isin(my_nums)].copy()
only_rep['Окончание (final_df)'] = only_rep['num_key'].map(fd_by_num['Окончание (final_df)'])

def rep_reason(r):
    end = r['Окончание (final_df)']
    if pd.notna(end) and end < as_of:
        return 'Закончился в августе, на 01.09 уже не действует'
    if r['inn_n'] and r['inn_n'] in my_inns:
        return 'Тот же клиент (ИНН) есть у меня под другим номером'
    if r['num_key'] not in fd_by_num.index:
        return 'Номера нет в final_df за август (номер записан иначе)'
    return 'В final_df действует на 01.09, а у меня нет - проверить (привязка P)'

only_my = mine[~mine['num_key'].isin(rep_nums)].copy()

def my_reason(r):
    if pd.notna(r['Дата открытия (мой)']) and r['Дата открытия (мой)'] >= as_of:
        return 'Открыт 01.09 или позже - после отчётного месяца'
    if r['inn_n'] and r['inn_n'] in rep_inns:
        return 'Тот же клиент (ИНН) есть в отчёте под другим номером'
    if r['num_key'] in fd_by_num.index:
        return 'Есть в final_df за август, нет в отчёте'
    return 'Нет ни в отчёте, ни в final_df за август'

only_rep['Причина'] = only_rep.apply(rep_reason, axis=1) if len(only_rep) else None
only_my['Причина'] = only_my.apply(my_reason, axis=1) if len(only_my) else None

def reason_table(df, title):
    t = df['Причина'].value_counts().rename_axis(title).reset_index(name='Договоров')
    t['Доля, %'] = (t['Договоров'] / max(len(df), 1) * 100).round(1)
    return pd.concat([t, pd.DataFrame([{title: 'Всего', 'Договоров': len(df), 'Доля, %': 100.0}])], ignore_index=True)

reasons_rep = reason_table(only_rep, 'Есть в отчёте за август, нет у меня на 01.09')
reasons_my = reason_table(only_my, 'Есть у меня на 01.09, нет в отчёте за август')
calc = len(rep_nums) - len(only_rep) + len(only_my)
math = pd.DataFrame([
    ('Номеров в отчёте за август', len(rep_nums)),
    ('− только в отчёте', len(only_rep)),
    ('+ только у меня', len(only_my)),
    ('= номеров у меня (расчёт)', calc),
    ('Номеров у меня (факт)', len(my_nums)),
    ('Сходится', 'да' if calc == len(my_nums) else 'нет'),
], columns=['Сверка количества', 'Значение'])
for t in (math, reasons_rep, reasons_my):
    display(t)


## 4. Что такое `id` в выгрузке ЦФТ

Для договоров, которые нашлись по номеру, сравниваю `id` ЦФТ с ID из других источников. Если совпадение близко к 100% - это тот же ID. Вторая колонка - какая доля `id` ЦФТ вообще встречается среди значений источника.


In [ ]:
base = cft[['num_key', 'id']].dropna()
checks = []

def compare(label, other, other_col):
    o = other[['num_key', other_col]].dropna().drop_duplicates('num_key').rename(columns={other_col: 'other_id'})
    m = base.merge(o, on='num_key')
    same = round(float((m['id'] == m['other_id']).mean() * 100), 1) if len(m) else None
    in_set = round(float(base['id'].isin(set(o['other_id'])).mean() * 100), 1) if len(base) else None
    checks.append((label, len(m), same, in_set))

compare('«ID договора» в отчёте за август (%s)' % ex_id, rep, 'id')
compare('agr_id в final_df (= ods.scd1_z_r2_ip_merchants.id)', fd, 'id')
compare('n_agr в final_df (ID договора в Альфе)', fd, 'n_agr_id')
compare('«ЦФТ ID» в моей выгрузке (ID клиента, c_cl_org)', mine, 'client_id')
id_checks = pd.DataFrame(checks, columns=['С чем сравниваю', 'Общих договоров по номеру', 'ID совпадает, %',
                                          'id ЦФТ встречается среди значений, %'])
display(id_checks)
top = id_checks.dropna(subset=['ID совпадает, %']).sort_values('ID совпадает, %', ascending=False)
if len(top) and top.iloc[0]['ID совпадает, %'] >= 90:
    print('Вывод: id в выгрузке ЦФТ = %s' % top.iloc[0]['С чем сравниваю'])
else:
    print('Вывод: id ЦФТ не совпал ни с одним известным ID - смотрим озеро ниже')

id_examples = (
    cft[['Номер договора (ЦФТ)', 'ID (ЦФТ)', 'Клиент (ЦФТ)', 'num_key']]
    .merge(rep[['num_key', 'ID договора (отчёт)']], on='num_key', how='left')
    .merge(fd[['num_key', 'agr_id (final_df)', 'n_agr (final_df)']].dropna(subset=['num_key'])
           .drop_duplicates('num_key'), on='num_key', how='left')
    .merge(mine[['num_key', 'ЦФТ ID']], on='num_key', how='left')
    .drop(columns='num_key')
    .head(200)
)
display(id_examples.head(15))


In [ ]:
lake_checks = pd.DataFrame()
if LOOKUP_LAKE:
    from rail_connectors.connection import connect
    if globals().get('imp') is None:
        LAKE_USER = input('Логин Impala: ').strip()
        LAKE_PASSWORD = getpass.getpass('Пароль Impala: ')
        imp = connect(
            to='IMPALA',
            extra_options={'db': 'sandbox_ai'},
            driver_args={'tez.queue.name': 'ai'},
            kerberos={
                'keytab_path': '/home/jovyan/test_requests/tech.keytab',
                'use_credentials': True,
                'update_keytab': True,
            },
            user_params={'user_name': LAKE_USER, 'password': LAKE_PASSWORD},
        )
        imp._init_connection()

    ids = cft['id'].dropna().drop_duplicates()
    sample = ids.sample(min(LAKE_SAMPLE, len(ids)), random_state=1)
    in_list = ', '.join("'%s'" % str(v).replace("'", "''") for v in sample)
    with imp:
        imp.execute('set MEM_LIMIT=16g')
        merch = imp.fetch(
            'SELECT cast(m.id AS string) AS id, cast(m.c_cl_org AS string) AS c_cl_org, m.c_name_in_pr '
            'FROM ods.scd1_z_r2_ip_merchants m WHERE cast(m.id AS string) IN (%s)' % in_list)
    merch.columns = [str(c).split('.')[-1].lower() for c in merch.columns]
    merch['id'] = merch['id'].map(norm_id)
    merch = merch.drop_duplicates('id')
    s = cft[cft['id'].isin(set(sample))].merge(merch, on='id', how='left')
    s['c_cl_org_n'] = s['c_cl_org'].map(norm_id)
    s['name_in_pr_key'] = s['c_name_in_pr'].map(norm_agr)
    my_client = mine.set_index('num_key')['client_id']
    s['my_client'] = s['num_key'].map(my_client)
    found = s['c_cl_org'].notna() | s['c_name_in_pr'].notna()
    lake_checks = pd.DataFrame([
        ('id ЦФТ в выборке', len(s)),
        ('найден как ods.scd1_z_r2_ip_merchants.id', int(found.sum())),
        ('  номер договора ЦФТ = c_name_in_pr этой записи', int((s['name_in_pr_key'] == s['num_key']).sum())),
        ('  есть у меня по номеру', int(s['my_client'].notna().sum())),
        ('  c_cl_org = мой «ЦФТ ID» (тот же клиент)', int((s['c_cl_org_n'] == s['my_client']).sum())),
    ], columns=['Проверка id по озеру', 'Договоров'])
    display(lake_checks)
    display(s.loc[found, ['Номер договора (ЦФТ)', 'ID (ЦФТ)', 'c_name_in_pr', 'c_cl_org', 'my_client']].head(15))
else:
    print('LOOKUP_LAKE = False - пропущено')


## 5. Excel

In [ ]:
def autofit(ws):
    for col_ in ws.columns:
        width = max(len(str(c.value)) if c.value is not None else 0 for c in col_)
        ws.column_dimensions[col_[0].column_letter].width = min(max(width + 2, 10), 70)

def dates(df):
    df = df.copy()
    for c in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[c]):
            df[c] = df[c].dt.date
    return df

OUT_FILE.parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(OUT_FILE, engine='openpyxl') as w:
    row = 0
    for t in (counts, overlap, math, reasons_rep, reasons_my, id_checks, lake_checks):
        if len(t):
            t.to_excel(w, index=False, sheet_name='Итог', startrow=row)
            row += len(t) + 2
    dates(only_rep[['Причина', 'Номер договора (отчёт)', 'ID договора (отчёт)', 'ИНН (отчёт)',
                    'Наименование (отчёт)', 'Окончание (final_df)']]).to_excel(
        w, index=False, sheet_name='Только в отчёте')
    dates(only_my[['Причина', 'Номер договора (мой)', 'ЦФТ ID', 'Дата открытия (мой)']]).to_excel(
        w, index=False, sheet_name='Только у меня')
    id_examples.to_excel(w, index=False, sheet_name='ID - примеры')
    for ws in w.book.worksheets:
        autofit(ws)
        if ws.title != 'Итог':
            ws.freeze_panes = 'A2'
print(OUT_FILE)
